# EYE Médicale — entraînement

CLIP, Gemma 2B et LoRA sur ROCOv2. Notebook du projet de juin 2025.
Les cellules d’entraînement nécessitent un environnement CUDA compatible avec Unsloth.


In [ ]:
# 1. Installation (run once)
!pip install -U "unsloth[kaggle-new]" transformers datasets bitsandbytes xformers trl peft

In [ ]:
# A — imports & environnement
import os, io, gc, torch, matplotlib.pyplot as plt
os.environ.update({
    "CUT_CROSS_ENTROPY_DISABLE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "BNB_4BIT_USE_FP16": "0",
    "CUDA_VISIBLE_DEVICES": "0",
})

from unsloth import FastLanguageModel, FastVisionModel   # ← AVANT transformers !
from transformers import (CLIPModel, CLIPProcessor,
                          get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)


In [ ]:
# B — Gemma + LoRA + CLIP + wrapper
############################################################################
# 1. Gemma 2 B 4-bit
############################################################################
gemma, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/gemma-2b-bnb-4bit",
    load_in_4bit      = True,
    device_map        = {"": 0} if device == "cuda" else {"": "cpu"},
    trust_remote_code = True,
)
tokenizer.add_special_tokens({"additional_special_tokens": ["<|image|>"]})
gemma.resize_token_embeddings(len(tokenizer))

############################################################################
# 2. LoRA léger
############################################################################
lora_cfg = LoraConfig(
    r              = 32,
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
)
gemma = get_peft_model(gemma, lora_cfg)
gemma.print_trainable_parameters()

############################################################################
# 3. CLIP vision
############################################################################
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

############################################################################
# 4. Wrapper multimodal
############################################################################
class GemmaMultimodal(torch.nn.Module):
    def __init__(self, llm, clip):
        super().__init__()
        self.llm   = llm
        self.clip  = clip
        self.proj  = torch.nn.Linear(
            clip.config.projection_dim, llm.config.hidden_size, bias=False
        ).to(device)

    def forward(self, input_ids, attention_mask, pixel_values=None, labels=None):
        txt_emb = self.llm.get_input_embeddings()(input_ids)
        with torch.no_grad():
            vis = self.clip.get_image_features(pixel_values=pixel_values)
        vis_emb = self.proj(vis).unsqueeze(1).to(txt_emb.dtype)
        img_tok = tokenizer.convert_tokens_to_ids("<|image|>")
        fused   = torch.where((input_ids == img_tok).unsqueeze(-1), vis_emb, txt_emb)
        return self.llm(
            inputs_embeds  = fused,
            attention_mask = attention_mask,
            labels         = labels,
        )

model = GemmaMultimodal(gemma, clip_model)
model.config = gemma.config            # pour save_pretrained

# patch Unsloth vision
os.environ["UNSLOTH_NO_COMPILE"] = "1"
FastVisionModel.for_training(model)


In [ ]:
# %% Préparation du dataset ROCO → format "messages" (sans PIL dans le Dataset)
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm
import re, io
from PIL import Image

# ---------- paramètres ----------
INST = "You are an expert radiologist. Describe the image precisely for a medical report."
N    = 16_000                                   # échantillon rapide

# ---------- helpers ----------
TAG_RE = re.compile(r"<[^>]+>")
def strip_tags(txt): return TAG_RE.sub("", txt).strip()

def pil_to_bytes(pil_img):
    """Convertit un PIL.Image en bytes PNG (picklable)."""
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return buf.getvalue()

# ---------- chargement (décode les images en PIL) ----------
raw = load_dataset("eltorio/ROCOv2-radiology", split=f"train[:{N}]")

examples = []
for rec in tqdm(raw, desc="Formattage", total=len(raw)):
    caption = strip_tags(rec["caption"])
    img     = rec["image"]               # PIL.Image

    # --> on stocke UNIQUEMENT les octets (pas l’objet PIL)
    img_bytes = pil_to_bytes(img)

    examples.append({
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text",  "text": INST},
                    {"type": "image", "image": {"bytes": img_bytes}},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": caption},
                ],
            },
        ]
    })

train_ds = Dataset.from_list(examples).with_format("python")
print("Loaded", len(train_ds), "examples")


In [ ]:
# %% 5. Collator custom corrigé (image + texte → batch prêt pour le modèle)
import io, torch
from PIL import Image
import torchvision.transforms as T

# pipeline d’augmentation (optionnel)
augment = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
])

def vision_sft_collate(batch, augment_images=True):
    pil_imgs, texts, prompts = [], [], []

    for ex in batch:
        # Chargement image
        imgf = ex["messages"][0]["content"][1]["image"]
        if isinstance(imgf, dict) and "bytes" in imgf:
            pil = Image.open(io.BytesIO(imgf["bytes"]))
        else:
            pil = Image.open(imgf)
        if pil.mode != "RGB":
            pil = pil.convert("RGB")
        if augment_images:
            pil = augment(pil)
        pil_imgs.append(pil)

        # Prompt et réponse
        prompts.append("<|image|> You are an expert radiologist. Describe the image precisely.")
        texts.append(ex["messages"][1]["content"][0]["text"])

    # Images
    px = clip_processor(images=pil_imgs, return_tensors="pt", padding=True).pixel_values

    # Tokenisation séparée
    enc_p = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
    enc_l = tokenizer(texts,   return_tensors="pt", padding=True, truncation=True)

    # Concat
    input_ids = torch.cat([enc_p.input_ids, enc_l.input_ids], dim=1)
    attention_mask = torch.cat([enc_p.attention_mask, enc_l.attention_mask], dim=1)

    labels = input_ids.clone()
    prompt_len = enc_p.input_ids.size(1)
    labels[:, :prompt_len] = -100
    labels = torch.where(input_ids == tokenizer.pad_token_id, -100, labels)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": px,
        "labels": labels,
    }


In [ ]:
# %% 6. Split & DataLoaders (modifié)
from torch.utils.data import DataLoader
from functools import partial

splits     = train_ds.train_test_split(test_size=0.1, seed=42)
train_part = splits["train"]
valid_part = splits["test"]
BATCH_SIZE = 8

train_loader = DataLoader(
    train_part,
    batch_size   = BATCH_SIZE,
    shuffle      = True,
    collate_fn   = vision_sft_collate,
    num_workers  = 2,        # régle ça plus bas si tu veux
    pin_memory   = False,    # <- désactive le pin_memory
)
valid_loader = DataLoader(
    valid_part,
    batch_size   = BATCH_SIZE,
    shuffle      = False,
    collate_fn   = partial(vision_sft_collate, augment_images=False),
    num_workers  = 2,        # idem
    pin_memory   = False,    # <- désactive le pin_memory
)
print(len(train_part), "train •", len(valid_part), "valid")


In [ ]:
# %% 8. Training Loop stable (AMP + grad-acc + cosine + early-stop)
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from torch.nn.utils import clip_grad_norm_
import torch.cuda.amp as amp
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# -------- hyper-paramètres --------
EPOCHS      = 5
GRAD_ACC    = 2                  # batch effectif = 8 × 2 = 16
LR          = 5e-6
W_DECAY     = 0.01
PATIENCE    = 3                  # early-stop
MAX_STEPS   = len(train_loader) * EPOCHS + 2000
WARMUP_STEPS= int(0.1 * MAX_STEPS)

# -------- optimizer & scheduler --------
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=LR, weight_decay=W_DECAY)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS
)
scaler    = amp.GradScaler()

# -------- training loop --------
best_val, wait, step = 1e9, 0, 0
loss_curve = []
model.train()
pbar = tqdm(total=MAX_STEPS, desc="steps")

while step < MAX_STEPS and wait < PATIENCE:
    for batch in train_loader:
        if step >= MAX_STEPS or wait >= PATIENCE:
            break

        batch = {k: v.to(device) for k, v in batch.items()}

        # ------ forward ------
        with amp.autocast():
            loss = model(**batch).loss / GRAD_ACC

        scaler.scale(loss).backward()

        # ------ optimiser tous les GRAD_ACC pas ------
        if (step + 1) % GRAD_ACC == 0:
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            loss_curve.append(loss.item() * GRAD_ACC)

        step += 1
        pbar.update(1)

    # ------ validation fin d’époque ------
    val_losses = []
    model.eval()
    with torch.no_grad():
        for vb in valid_loader:
            vb = {k: v.to(device) for k, v in vb.items()}
            val_losses.append(model(**vb).loss.item())
    val = sum(val_losses) / len(val_losses)
    print(f"\nEpoch terminé — valid loss = {val:.4f}")

    if val < best_val:
        best_val, wait = val, 0
        torch.save(model.state_dict(), "best_adapter.pt")
    else:
        wait += 1
    model.train()

pbar.close()

# -------- courbe de loss --------
plt.plot(loss_curve)
plt.xlabel("Optimizer steps")
plt.ylabel("Training loss")
plt.title("Loss curve")
plt.show()


In [ ]:
# Inférence sur une image
import io, re, torch
from PIL import Image
import matplotlib.pyplot as plt

def clean_repetition(text: str) -> str:
    plain = re.sub(r"<[^>]+>", "", text).strip()
    parts = re.split(r'([\.!?]\s|\n)', plain)
    seen, out = set(), []
    for i in range(0, len(parts), 2):
        sent = parts[i].strip()
        if not sent:
            continue
        sep = parts[i+1] if i+1 < len(parts) else ""
        full = f"{sent}{sep}".strip()
        if full not in seen:
            seen.add(full)
            out.append(full)
    return " ".join(out)

# Choisissez l'indice de l'exemple à tester
sample_idx = 1

# 1) Récupérer l’exemple et afficher l’image
sample = train_ds[sample_idx]
raw    = sample["messages"][0]["content"][1]["image"]
if isinstance(raw, dict):
    img = Image.open(io.BytesIO(raw["bytes"])).convert("RGB")
elif not isinstance(raw, Image.Image):
    img = Image.open(raw).convert("RGB")
else:
    img = raw

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis("off")
plt.title(f"Sample #{sample_idx}")
plt.show()

# 2) Préparer le prompt
inst = "You are a medical radiologist. Please analyze the image and describe it precisely."
prompt = "<|image|> " + inst
print("Prompt →", inst)

# 3) Encoder image + prompt
px = clip_processor(images=img, return_tensors="pt", padding=True).pixel_values.to(device)
enc = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
input_ids      = enc.input_ids
attention_mask = enc.attention_mask

# 4) Génération auto (pas-à-pas)
model.eval()
gen_ids = []
max_new_tokens = 150
with torch.no_grad():
    for _ in range(max_new_tokens):
        out = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            pixel_values   = px,
        )
        nxt = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
        if nxt.item() == tokenizer.eos_token_id:
            break
        gen_ids.append(nxt.item())
        nxt = nxt.to(device)  # <-- déplace bien sur le même device !
        input_ids      = torch.cat([input_ids,      nxt], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(nxt, device=device)], dim=1)

# 5) Décodage et nettoyage
raw_text   = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
clean_text = clean_repetition(raw_text)

print("\n→ Légende brute   :", raw_text)
print("→ Légende nettoyée:", clean_text)


## Export pour l’API
Cet export conserve les adaptateurs, le tokenizer et la projection visuelle entraînée.


In [ ]:
from pathlib import Path
import json

export_dir = Path("model/eye-medicale")
export_dir.mkdir(parents=True, exist_ok=True)
model.llm.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)
torch.save(model.proj.state_dict(), export_dir / "vision_projector.pt")
(export_dir / "multimodal_config.json").write_text(json.dumps({
    "base_model": "unsloth/gemma-2b-bnb-4bit",
    "clip_model": "openai/clip-vit-base-patch32",
    "image_token": "<|image|>",
}, indent=2))
print("Modèle exporté dans", export_dir)
